# 09｜分群與表示應用

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/09_分群與表示應用.md`，目前 50 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

Lloyd 迭代、逐群輪廓、代表樣本、GMM／EM、階層樹與 DBSCAN。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 09-03 | 8、9、11、12 | 1. K-means 兩步交替與手算驗算 |
| 09-04 | 6、7、15、16 | 2. K-means++、多次初始化與局部解 |
| 09-05 | 3、19、20、21、22、24 | 3. 肘部法與輪廓分數，以及輪廓係數手算 |
| 09-06 | 25、26 | 4. K-means 的限制：橢圓、密度與群大小差異 |
| 09-07 | 5、28、29、30 | 5. 顏色量化：K-means 做顏色分群 |
| 09-08 | 32、34、35、36、38、49 | 6. 少量標註：先標代表樣本，再傳播 |
| 09-09 | 4、6、7、8、9、10、12、13、15、16、17、18、19、50 | 7. 看見指派、更新、初始化與 k |
| 09-10 | 21、22、23、24、26、50 | 8. 每群輪廓圖與尺度 |
| 09-11 | 5、31、32、33、34、35、36、37、38、50 | 9. 代表樣本、可信範圍與距離特徵 |
| 09-12 | 40、41、42 | 10. GMM 軟分群與 EM 更新 |
| 09-13 | 43、44、45 | 11. 階層式分群、linkage 與樹狀圖 |
| 09-14 | 46、47、48 | 12. DBSCAN 的核心點、密度參數與雜訊 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 09-01｜投影片 09：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'scipy': 'scipy>=1.11,<2', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 09-02｜投影片 09：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 1. K-means 兩步交替與手算驗算

指派：每點到最近中心；更新：中心移到群平均。對應投影片活動：一維資料 `[0,1,4,5]`、初始中心 `[0,4]`。

In [ ]:
# 程式 09-03｜投影片 09：8、9、11、12 頁
def assign(x, centers):
    return np.argmin(np.abs(np.asarray(x)[:, None] - np.asarray(centers)[None]), axis=1)
def inertia(x, centers, labels):
    return float(np.sum((np.asarray(x) - np.asarray(centers)[labels]) ** 2))

x1 = np.array([0., 1., 4., 5.])
centers0 = np.array([0., 4.])
lab0 = assign(x1, centers0)
inertia_before = inertia(x1, centers0, lab0)
cen1 = np.array([x1[lab0 == k].mean() for k in range(2)])   # 更新步
lab1 = assign(x1, cen1)
inertia_after = inertia(x1, cen1, lab1)
assert lab0.tolist() == [0, 0, 1, 1] and np.isclose(inertia_before, 2.0)
assert np.allclose(cen1, [0.5, 4.5]) and np.isclose(inertia_after, 1.0)
assert lab1.tolist() == lab0.tolist()      # 指派不再改變 -> 收斂
pd.Series({'initial_labels': lab0.tolist(), 'inertia_before': inertia_before,
           'updated_centers': cen1.tolist(), 'inertia_after': inertia_after})

## 2. K-means++、多次初始化與局部解

不同初始化可能收斂到不同局部解。`init="k-means++"` 讓初始中心分散，`n_init` 取多次最佳（依 inertia）。

In [ ]:
# 程式 09-04｜投影片 09：6、7、15、16 頁
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
centers = np.array([[-6, -6], [0, -6], [6, -6], [-3, 4], [4, 4]])
X, y_true = make_blobs(n_samples=800, centers=centers, cluster_std=0.9, random_state=SEED)
runs = []
for seed in range(6):
    km = KMeans(n_clusters=5, init='random', n_init=1, random_state=seed).fit(X)
    runs.append(km.inertia_)
best = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=SEED).fit(X)
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                     np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
zz = best.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=.25, cmap='tab10')
plt.scatter(X[:, 0], X[:, 1], c=best.labels_, cmap='tab10', s=10)
plt.scatter(best.cluster_centers_[:, 0], best.cluster_centers_[:, 1], c='k', marker='x', s=120)
plt.title('K-means++ Voronoi regions')
savefig('14_kmeans_voronoi')
print('single random-init inertia range:', round(min(runs), 1), '-', round(max(runs), 1),
      ' | k-means++ x10 inertia:', round(best.inertia_, 1))

## 3. 肘部法與輪廓分數，以及輪廓係數手算

肘部看 inertia 改善開始遞減的位置；輪廓分數把群內距離 $a$ 與最近他群距離 $b$ 一起比較，$s=(b-a)/\max(a,b)$。對應投影片活動的兩點。

In [ ]:
# 程式 09-05｜投影片 09：3、19、20、21、22、24 頁
from sklearn.metrics import silhouette_score
def silhouette(a, b):
    return (b - a) / max(a, b)
assert np.isclose(silhouette(2, 5), 0.6) and np.isclose(silhouette(4, 3), -0.25)

ks = range(2, 9)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X).inertia_ for k in ks]
sils = [silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X)) for k in ks]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(ks), inertias, 'o-'); axes[0].set(xlabel='k', ylabel='inertia', title='elbow')
axes[1].plot(list(ks), sils, 'o-'); axes[1].set(xlabel='k', ylabel='mean silhouette', title='silhouette')
savefig('14_kmeans_selection')
best_k = list(ks)[int(np.argmax(sils))]
assert best_k == 5                                   # 資料本來 5 群
print('silhouette picks k =', best_k, ' | hand check: s(a=2,b=5)=0.6, s(a=4,b=3)=-0.25')

## 4. K-means 的限制：橢圓、密度與群大小差異

K-means 偏好相近大小、球狀、密度相近的群。縮放前要先想清楚「相似」的定義。

In [ ]:
# 程式 09-06｜投影片 09：25、26 頁
rng = np.random.default_rng(SEED)
blob_a = rng.normal([0, 0], [3.0, 0.4], (400, 2))
blob_b = rng.normal([6, 0], [0.4, 0.4], (60, 2))
Xe = np.vstack([blob_a, blob_b])
km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(Xe)
plt.scatter(Xe[:, 0], Xe[:, 1], c=km.labels_, cmap='coolwarm', s=10)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c='k', marker='x', s=120)
plt.title('elongated + tiny cluster: K-means splits the big blob')
savefig('14_kmeans_limits')
small_recovered = np.bincount(km.labels_[-60:]).max() / 60
print('fraction of the true tiny cluster kept in one K-means label:', round(small_recovered, 2))

## 5. 顏色量化：K-means 做顏色分群

這是顏色量化，不會辨識物體或語意。把每個像素當成 RGB 三維點，用 K 個群中心取代所有像素顏色。使用 scikit-learn 內建 `flower.jpg`。

In [ ]:
# 程式 09-07｜投影片 09：5、28、29、30 頁
from sklearn.datasets import load_sample_image
image = load_sample_image('flower.jpg')
pixels = (image / 255.0).reshape(-1, 3)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(image); axes[0].set_title(f'original ({image.shape[0]}x{image.shape[1]})'); axes[0].axis('off')
for ax, n in zip(axes[1:], [8, 4, 2]):
    km = KMeans(n_clusters=n, n_init=3, random_state=SEED).fit(pixels[::5])   # 抽樣加速
    seg = km.cluster_centers_[km.predict(pixels)].reshape(image.shape)
    ax.imshow(seg); ax.set_title(f'{n} colors'); ax.axis('off')
savefig('14_color_quantization')
km8 = KMeans(n_clusters=8, n_init=3, random_state=SEED).fit(pixels[::5])
print('unique colors: original', len(np.unique(pixels, axis=0)), '-> quantized to', km8.n_clusters)

## 6. 少量標註：先標代表樣本，再傳播

digits 只給 50 個標籤預算。比較「隨機 50 個」與「每群挑最接近中心的一張」，再把群中心標籤傳播給整群。

In [ ]:
# 程式 09-08｜投影片 09：32、34、35、36、38、49 頁
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(digits.data, digits.target, test_size=.3, stratify=digits.target, random_state=SEED)
k = 50
km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xtr)
rep_idx = np.argmin(km.transform(Xtr), axis=0)          # 每群最接近中心的樣本
rep_labels = ytr[rep_idx]

def score(mask_or_idx, y_used):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    clf.fit(Xtr[mask_or_idx], y_used)
    return clf.score(Xte, yte)

rng = np.random.default_rng(SEED)
rand_idx = rng.choice(len(Xtr), k, replace=False)
acc_random = score(rand_idx, ytr[rand_idx])
acc_representative = score(rep_idx, rep_labels)
propagated = rep_labels[km.labels_]                     # 把代表標籤傳播給整群
acc_propagated = score(np.arange(len(Xtr)), propagated)
report('clustering', {'kmeans_1d': {'inertia_before': 2.0, 'inertia_after': 1.0},
       'silhouette_best_k': int(best_k), 'label_budget': k,
       'accuracy_random_labels': acc_random, 'accuracy_representative_labels': acc_representative,
       'accuracy_after_propagation': acc_propagated})
# 代表樣本不保證優於隨機標記；單次比較不是穩健優越性的證據。
pd.Series({'random_50_labels': acc_random, 'representative_50_labels': acc_representative,
           'propagate_to_full_train': acc_propagated})

## 7. 看見指派、更新、初始化與 k

逐步展示 Lloyd 更新；固定群編號只是繪圖方便。K-means++ 的下一中心機率與最近中心距離平方成正比；MiniBatchKMeans 用較小批次近似更新。

In [ ]:
# 程式 09-09｜投影片 09：4、6、7、8、9、10、12、13、15、16、17、18、19、50 頁
from sklearn.datasets import load_iris
from sklearn.cluster import MiniBatchKMeans
iris9 = load_iris(); fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(*iris9.data[:, 2:4].T, s=12); axes[0].set_title('Iris: labels hidden')
axes[1].scatter(*iris9.data[:, 2:4].T, c=iris9.target, s=12); axes[1].set_title('Labels revealed for interpretation')
savefig('c09_iris_unlabeled')
cent9 = X[[0, 10, 20, 30, 40]].copy(); previous9 = np.inf
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax in axes:
    distances9 = np.linalg.norm(X[:, None] - cent9[None], axis=2)
    labels9 = distances9.argmin(axis=1); inertia9 = np.sum(distances9[np.arange(len(X)), labels9] ** 2)
    assert inertia9 <= previous9 + 1e-8; previous9 = inertia9
    ax.scatter(*X.T, c=labels9, s=5, cmap='tab10'); ax.scatter(*cent9.T, c='k', marker='x', s=70)
    ax.set_title(f'Inertia={inertia9:.0f}')
    cent9 = np.array([X[labels9==j].mean(0) if np.any(labels9==j) else cent9[j] for j in range(5)])
savefig('c09_lloyd_iterations')
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (k9, seed9) in zip(axes, [(5, 0), (5, 1), (3, 42), (7, 42)]):
    model9 = KMeans(k9, init='random', n_init=1, random_state=seed9).fit(X)
    ax.scatter(*X.T, c=model9.labels_, s=6, cmap='tab10'); ax.set_title(f'k={k9}, seed={seed9}\nJ={model9.inertia_:.0f}')
savefig('c09_initialization_and_k')
D9 = best.transform(X)
assert np.allclose(-best.score(X), np.min(D9, axis=1).dot(np.min(D9, axis=1)))
D2 = (np.array([0., 1., 4., 5.]) - 0.) ** 2; probabilities9 = D2 / D2.sum()
print('Next-center probabilities for first center 0:', probabilities9, 'distance feature shape:', D9.shape)
print('MiniBatch / full inertia:', MiniBatchKMeans(5, batch_size=80, n_init=3, random_state=SEED).fit(X).inertia_, best.inertia_)

## 8. 每群輪廓圖與尺度

平均分數可能掩蓋負值樣本或過小的群。縮放會改變距離定義，必須配合特徵意義。

In [ ]:
# 程式 09-10｜投影片 09：21、22、23、24、26、50 頁
from sklearn.metrics import silhouette_samples
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, k9 in zip(axes, [3, 5]):
    labs9 = KMeans(k9, n_init=10, random_state=SEED).fit_predict(X)
    values9 = silhouette_samples(X, labs9); start9 = 0
    for j in range(k9):
        one9 = np.sort(values9[labs9 == j]); ax.fill_betweenx(np.arange(start9, start9+len(one9)), 0, one9)
        start9 += len(one9) + 10
    ax.axvline(values9.mean(), c='r', ls='--'); ax.set(xlabel='silhouette', ylabel='samples ordered by cluster', title=f'k={k9}')
savefig('c09_silhouette_profiles')
raw9 = Xe * [100., 1.]
scaled9 = StandardScaler().fit_transform(raw9)
print('Feature std before / after:', raw9.std(0), scaled9.std(0))

## 9. 代表樣本、可信範圍與距離特徵

預先固定保留每群距中心最近的 20% 作標籤傳播，比較同一測試集；測試集不拿來挑百分位。主動學習僅列下一批標註需求，尚未取得標籤，不能宣稱已提升成效。

In [ ]:
# 程式 09-11｜投影片 09：5、31、32、33、34、35、36、37、38、50 頁
fig, axes = plt.subplots(5, 10, figsize=(11, 6))
for j, ax in enumerate(axes.flat):
    ax.imshow(Xtr[rep_idx[j]].reshape(8,8), cmap='binary'); ax.set_title(f'{j}: {rep_labels[j]}', fontsize=9); ax.axis('off')
savefig('c09_representative_digits')
assigned_dist9 = km.transform(Xtr)[np.arange(len(Xtr)), km.labels_]
keep9 = np.zeros(len(Xtr), dtype=bool)
for j in range(k):
    member9 = km.labels_ == j; keep9[member9] = assigned_dist9[member9] <= np.percentile(assigned_dist9[member9], 20)
keep9[rep_idx] = True
print('Partial propagation kept / accuracy:', keep9.sum(), score(keep9, propagated[keep9]))
# 距中心特徵必須在每個 CV 訓練折重新擬合 K-means
from sklearn.model_selection import cross_val_score
repr_model9 = make_pipeline(StandardScaler(), KMeans(20, n_init=3, random_state=SEED), StandardScaler(), LogisticRegression(max_iter=2000))
print('Distance-representation training CV:', cross_val_score(repr_model9, Xtr, ytr, cv=3).mean())
active9 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr[rep_idx], rep_labels)
pool9 = np.setdiff1d(np.arange(len(Xtr)), rep_idx)
conf9 = active9.predict_proba(Xtr[pool9]).max(axis=1)
request9 = pool9[np.argsort(conf9)[:5]]
assert not np.intersect1d(request9, rep_idx).size
print('Next 5 annotation requests (training pool indices):', request9)

## 10. GMM 軟分群與 EM 更新

同一筆資料對每個高斯成分都有責任值。比較硬群號、最大責任值與 BIC，不能把成分編號當成真實類別。

In [ ]:
# 程式 09-12｜投影片 09：40、41、42 頁
from sklearn.mixture import GaussianMixture
rng9 = np.random.default_rng(SEED)
Xg9 = np.vstack([
    rng9.multivariate_normal([-2., 0.], [[1.3, .8], [.8, .9]], 350),
    rng9.multivariate_normal([2., 1.], [[.7, -.45], [-.45, 1.2]], 350)])
gm9 = GaussianMixture(n_components=2, covariance_type='full', n_init=5,
                      reg_covar=1e-6, random_state=SEED).fit(Xg9)
resp9 = gm9.predict_proba(Xg9); hard9 = resp9.argmax(axis=1)
assert np.allclose(resp9.sum(axis=1), 1) and gm9.converged_
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(*Xg9.T, c=hard9, s=8, cmap='viridis'); axes[0].scatter(*gm9.means_.T, c='red', marker='x', s=100)
axes[0].set_title('Hard label from largest responsibility')
im9 = axes[1].scatter(*Xg9.T, c=resp9.max(axis=1), s=8, cmap='magma', vmin=.5, vmax=1)
axes[1].set_title('Largest responsibility'); fig.colorbar(im9, ax=axes[1])
savefig('c09_gmm_soft_clusters')
bic9 = {k9: GaussianMixture(k9, n_init=3, random_state=SEED).fit(Xg9).bic(Xg9) for k9 in range(1, 6)}
print('BIC by component count (lower is better):', {k: round(v, 1) for k, v in bic9.items()})

## 11. 階層式分群、linkage 與樹狀圖

同一份抽樣資料先建立凝聚樹，再以固定群數切群。樹狀圖的合併高度與葉順序要分開解讀。

In [ ]:
# 程式 09-13｜投影片 09：43、44、45 頁
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering
Xh9 = Xg9[::10]
Z9 = linkage(Xh9, method='ward', metric='euclidean')
hc9 = AgglomerativeClustering(n_clusters=2, linkage='ward').fit(Xh9)
assert Z9.shape == (len(Xh9) - 1, 4)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dendrogram(Z9, truncate_mode='lastp', p=20, ax=axes[0])
axes[0].set(xlabel='leaf or merged cluster', ylabel='merge distance', title='Ward dendrogram')
axes[1].scatter(*Xh9.T, c=hc9.labels_, s=18, cmap='viridis'); axes[1].set_title('Cut to 2 clusters')
savefig('c09_hierarchical_dendrogram')

## 12. DBSCAN 的核心點、密度參數與雜訊

彎月資料先標準化，再檢查核心、邊界與 `-1` 雜訊。`eps` 與 `min_samples` 的結果只對目前尺度與距離定義成立。

In [ ]:
# 程式 09-14｜投影片 09：46、47、48 頁
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
Xd9, _ = make_moons(n_samples=300, noise=0.08, random_state=SEED)
Xd9 = StandardScaler().fit_transform(Xd9)
db9 = DBSCAN(eps=0.24, min_samples=5).fit(Xd9)
core9 = np.zeros(len(Xd9), dtype=bool); core9[db9.core_sample_indices_] = True
border9 = (db9.labels_ != -1) & ~core9; noise9 = db9.labels_ == -1
n_clusters9 = len(set(db9.labels_) - {-1})
assert np.all(db9.labels_[noise9] == -1)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(*Xd9[core9].T, c=db9.labels_[core9], s=28, cmap='tab10', label='core')
ax.scatter(*Xd9[border9].T, c=db9.labels_[border9], s=20, cmap='tab10', marker='s', label='border')
ax.scatter(*Xd9[noise9].T, c='black', s=24, marker='x', label='noise')
ax.set_title(f'DBSCAN: {n_clusters9} clusters, {noise9.sum()} noise points'); ax.legend()
savefig('c09_dbscan_moons')
print({'clusters': n_clusters9, 'core': int(core9.sum()), 'border': int(border9.sum()), 'noise': int(noise9.sum())})

## 練習與參考答案

- 群編號 `2` 比 `1` 大嗎？**否，編號沒有大小或固定語義。**
- 平均輪廓分數很高就代表分群成功嗎？**否，要看每群分布與少數群的負值，以及業務意義。**
- 一群只有一個樣本怎麼算輪廓？**沒有同群其他點可平均，通常設 $s=0$。**
- 顏色量化只用 RGB，缺了什麼？**空間位置與語義；相鄰但不同物件的像素可能同群。**
- GMM 的 `predict_proba` 與 K-means 群號有何不同？**前者提供每個成分的責任值，後者是最近中心的硬指派。**
- DBSCAN 的 `-1` 表示什麼？**目前 `eps`、`min_samples` 與距離尺度下未被任何密度群吸收的雜訊點。**